# GenInherit-LLM: Colab Free-Tier Progressive Growth Ladder
### Stage 1: ~30M (45 mins) ➔ Stage 2: ~55M (1 hr) ➔ Stage 3: ~82M (1.5 hrs)

This notebook trains GenInherit in bite-sized sessions that **never trigger Google Colab daily GPU limits**:
1. **Stage 1 (4 layers, 28.1M params)**: Trains base grammar and Mendelian rules in **~45 minutes**.
2. **Stage 2 (8 layers, 52.9M params)**: Doubles depth via identity growth operator (`grow.py`) with zero loss shock.
3. **Stage 3 (12 layers, 82.0M params)**: Completes the full clinical foundation model.
4. **Zero-Loss Protection**: Checkpoints save continuously to Google Drive every 150 steps.

### Step 1: Mount Google Drive & Environment Check
Connects your Drive where Phase 1 shards are stored and checks your GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, torch
WORKSPACE_DRIVE = '/content/drive/MyDrive/geninherit'
CODE_DIR = f'{WORKSPACE_DRIVE}/code'
SHARDS_DIR = f'{WORKSPACE_DRIVE}/data/shards'

os.environ['PYTHONPATH'] = f"{CODE_DIR}:{os.environ.get('PYTHONPATH', '')}"

print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

### Step 2: Verify Phase 1 Shards & Install Dependencies

In [ ]:
!pip install -q pyyaml safetensors

shards = glob.glob(f'{SHARDS_DIR}/*.pt')
print(f'✓ Found {len(shards)} binary dataset shards in {SHARDS_DIR}. Ready!')

### Step 3: Train Stage 1 (~30 Million Parameters)
- **Configuration**: 4 Layers, Hidden Dim 768, 8 Query Heads, 2 KV Heads (GQA), 28.1M Parameters.
- **Execution Time**: **~45 to 50 minutes**.
- **Target**: 150 Million tokens.
- Checkpoints save to: `runs/r00_30m/latest.pt`

In [ ]:
!python /content/drive/MyDrive/geninherit/code/train_base.py \
    --config /content/drive/MyDrive/geninherit/code/configs/r00_30m.yaml \
    --data_dir /content/drive/MyDrive/geninherit/data/shards \
    --out_dir /content/drive/MyDrive/geninherit/runs/r00_30m \
    --session_hours 2.0

### Step 4: Evaluate Stage 1 Quality Gates
Runs validation on held-out test families (IID and Zero-Shot Compositional) to evaluate the 30M model.

In [ ]:
!python /content/drive/MyDrive/geninherit/code/run_eval_gates.py \
    --checkpoint /content/drive/MyDrive/geninherit/runs/r00_30m/geninherit_r00_30m_end.pt \
    --data_dir /content/drive/MyDrive/geninherit/data/shards

### Step 5: Grow to Stage 2 (~55 Million Parameters)
Uses `grow.py` to double depth from 4 layers to 8 layers with zero loss shock!

In [ ]:
!python /content/drive/MyDrive/geninherit/code/grow_and_train.py \
    --source_ckpt /content/drive/MyDrive/geninherit/runs/r00_30m/geninherit_r00_30m_end.pt \
    --target_config /content/drive/MyDrive/geninherit/code/configs/r01_55m.yaml \
    --data_dir /content/drive/MyDrive/geninherit/data/shards \
    --out_dir /content/drive/MyDrive/geninherit/runs/r01_55m